# Lab 04 — Quantum Teleportation

Alice teleports an unknown qubit state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$
to Bob using one shared Bell pair and two classical bits.

| Qubit | Role |
|---|---|
| `q[0]` | Alice's input $|\psi\rangle$ |
| `q[1]` | Alice's half of the Bell pair |
| `q[2]` | Bob's half of the Bell pair (receives $|\psi\rangle$) |

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(ROOT))

## Parameters
Edit these and re-run the notebook. Amplitudes are normalized automatically.

In [ ]:
ALPHA, BETA = 0.6, 0.8   # input state alpha|0> + beta|1>
SHOTS = 1024             # shots for the statistical check
SEED = 7                 # simulator seed (for reproducible results)

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import StatePreparation, UnitaryGate
from qiskit.quantum_info import Operator, Statevector
from qiskit_aer import AerSimulator

from common.qlab import normalize, format_state, qubit_state, fidelity, save_circuit

alpha, beta = normalize(ALPHA, BETA)
print("Alice's input state:", format_state(alpha, beta))

## Step 1 — Prepare input, Bell pair, and Bell-basis transformation

In [ ]:
def teleport_unitary_part(alpha, beta):
    q = QuantumRegister(3, "q")
    qc = QuantumCircuit(q)

    qc.prepare_state([alpha, beta], q[0])
    qc.barrier(label="Input State Preparation")

    qc.h(q[1])
    qc.cx(q[1], q[2])
    qc.barrier(label="Entanglement Preparation")

    qc.cx(q[0], q[1])
    qc.h(q[0])
    qc.barrier(label="Bell Basis Transformation")

    return qc, q


qc_pre, q = teleport_unitary_part(alpha, beta)
sv = Statevector(qc_pre)

## Step 2 — Bob's state for every possible measurement outcome

Before Alice measures, the 3-qubit state is
$\frac{1}{2}\sum_{m_0 m_1} |m_0 m_1\rangle \otimes X^{m_1} Z^{m_0} |\psi\rangle$.
For each outcome ($m_0$ from `q[0]`, $m_1$ from `q[1]`) we project onto it,
read Bob's qubit, and apply the correction $Z^{m_0} X^{m_1}$.

In [ ]:
outcome_fidelities = {}

for m0 in (0, 1):
    for m1 in (0, 1):
        base = m0 + 2 * m1                  # little-endian index of q0=m0, q1=m1, q2=0
        bob_alpha, bob_beta = normalize(sv.data[base], sv.data[base + 4])
        before = format_state(bob_alpha, bob_beta)

        corrections = []
        if m1 == 1:
            bob_alpha, bob_beta = bob_beta, bob_alpha
            corrections.append("X")
        if m0 == 1:
            bob_beta = -bob_beta
            corrections.append("Z")

        f = fidelity([alpha, beta], [bob_alpha, bob_beta])
        outcome_fidelities[f"{m0}{m1}"] = f

        print(f"m0 m1 = {m0}{m1} | Bob before: {before} | "
              f"corrections: {' then '.join(corrections) or 'none':<10} | "
              f"after: {format_state(bob_alpha, bob_beta)} | fidelity = {f:.6f}")

## Step 3 — Full dynamic circuit: measure, classically controlled corrections

In [ ]:
qc, q = teleport_unitary_part(alpha, beta)
c = ClassicalRegister(2, "c")
qc.add_register(c)

qc.barrier(label="Measurement Preparation")
qc.measure(q[0], c[0])
qc.measure(q[1], c[1])
qc.barrier(label="Alice's Measurement Results")

with qc.if_test((c[1], 1)):
    qc.x(q[2])
with qc.if_test((c[0], 1)):
    qc.z(q[2])
qc.barrier(label="Bob's Error Correction")

# q[0] and q[1] are measured and no longer needed; q[2] holds Bob's state.
# Resetting q[2] would destroy the teleported state.
qc.reset(q[0])
qc.reset(q[1])
qc.barrier(label="Alice's Qubits Reset - Available for Reuse")
qc.barrier(label="Recovered State")

save_circuit(qc, "circuit.png", scale=1.0);

## Step 4 — Simulate one run and inspect Bob's qubit

In [ ]:
backend = AerSimulator(method="statevector", seed_simulator=SEED)

run = qc.copy()
run.save_statevector(label="final")
result = backend.run(transpile(run, backend), shots=1).result()

measured = next(iter(result.get_counts()))
bob_alpha, bob_beta = qubit_state(result.data(0)["final"], qubit=2)
run_fidelity = fidelity([alpha, beta], [bob_alpha, bob_beta])

print("Alice measured (c1 c0):", measured)
print("Alice's input:", format_state(alpha, beta))
print("Bob's output :", format_state(bob_alpha, bob_beta))
print("Fidelity     :", round(run_fidelity, 10))

## Step 5 — Statistical check over many shots

Apply the inverse of the state preparation to Bob's qubit and measure it.
If teleportation worked, Bob's qubit returns to $|0\rangle$ in **every** shot,
whatever Alice measured.

In [ ]:
check = qc.copy()
b = ClassicalRegister(1, "bob")
check.add_register(b)
# Undo the preparation with the explicit adjoint unitary U_psi^dagger
# (qiskit-aer 0.18 crashes on StatePreparation(...).inverse()).
undo = Operator(StatePreparation([alpha, beta])).adjoint()
check.append(UnitaryGate(undo, label="U_psi†"), [q[2]])
check.measure(q[2], b[0])

counts = backend.run(transpile(check, backend), shots=SHOTS).result().get_counts()
bob_ones = sum(n for key, n in counts.items() if key.split()[0] == "1")

print("Counts (bob c1c0):", counts)
print(f"Shots where Bob's qubit was NOT restored: {bob_ones} / {SHOTS}")

## Verification

In [ ]:
assert all(np.isclose(f, 1.0) for f in outcome_fidelities.values()), outcome_fidelities
assert np.isclose(run_fidelity, 1.0), run_fidelity
assert bob_ones == 0, counts
print("Teleportation successful for all outcomes!")

In [ ]:
qc.draw(output="mpl", fold=-1, scale=1.0)